# Import the model architecture

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import AutoTokenizer
import os

In [2]:
MAX_TARGET_LEN = 512

In [3]:
df = pd.read_csv("model1_ready_data.csv")

input_lengths = df['input_text'].str.len()
target_lengths = df['target_text'].str.len()

max_input = input_lengths.max()
max_target = target_lengths.max()

print(f"Max Input Length: {max_input}")
print(f"Max Target Length: {max_target}")

# To see the distribution (useful to see if a few outliers are skewing the max)
print(target_lengths.describe())

Max Input Length: 2099
Max Target Length: 5812
count    96108.000000
mean       267.518032
std         65.159268
min         31.000000
25%        235.000000
50%        256.000000
75%        284.000000
max       5812.000000
Name: target_text, dtype: float64


In [ ]:
import torch
import torch.nn as nn

# 1. Define the vocabulary
chars = " abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'|+-<>={},"

# 2. Build char_to_ix with the i+4 offset
char_to_ix = {ch: i+4 for i, ch in enumerate(chars)}
char_to_ix["<PAD>"] = 0
char_to_ix["<UNK>"] = 1
char_to_ix["<SOS>"] = 2
char_to_ix["<EOS>"] = 3

# 3. Build the reverse mapping (ix_to_char)
ix_to_char = {i: ch for ch, i in char_to_ix.items()}

vocab_size = len(char_to_ix)

print(f"Vocab Size: {vocab_size}")
print(f"First few mappings: {list(char_to_ix.items())[:10]}")
print(f"Index 2: {ix_to_char[2]}")  # Should be <SOS>
print(f"Index 4: {ix_to_char[4]}")  # Should be ' ' (the first char in the string)

Vocab Size: 77
First few mappings: [(' ', 4), ('a', 5), ('b', 6), ('c', 7), ('d', 8), ('e', 9), ('f', 10), ('g', 11), ('h', 12), ('i', 13)]
Index 2: <SOS>
Index 4:  


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim) # To compress bidirectional states
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs: [src_len, batch_size, hid_dim * 2]
        # hidden: [n_layers * 2, batch_size, hid_dim]
        
        # Concat the bidirectional hidden states
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))
        
        return outputs, (hidden.unsqueeze(0), cell.unsqueeze(0))

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear((hid_dim * 2) + hid_dim, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [1, batch_size, hid_dim]
        # encoder_outputs: [src_len, batch_size, hid_dim * 2]
        src_len = encoder_outputs.shape[0]
        h = hidden.repeat(src_len, 1, 1)
        energy = torch.tanh(self.attn(torch.cat((h, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=0)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM((hid_dim * 2) + emb_dim, hid_dim, n_layers, dropout=dropout)
        self.out = nn.Linear((hid_dim * 2) + hid_dim + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch_size]
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        
        # Calculate attention weights
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        # a: [batch_size, 1, src_len]
        a = a.permute(2, 1, 0)
        
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        # print(encoder_outputs.shape)
        # print(a.shape)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)
        # weighted: [1, batch_size, hid_dim * 2]
        
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        prediction = self.out(torch.cat((output, weighted, embedded), dim=2)).squeeze(0)
        return prediction, hidden, cell

In [6]:
# 1. Configuration (Must match the training notebook exactly)
EMBEDDING_DIM = 128
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.2 # Dropout doesn't matter for inference, but the class needs it

# 2. Initialize Models
attn_eval = Attention(HIDDEN_DIM)
enc_eval = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).cuda()
dec_eval = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, attn_eval).cuda()

# 3. Load the Weights
# Epoch 5 gave the best results upon comparison, model likely overfit after that
# as the training loss still went down.

enc_eval.load_state_dict(torch.load('encoder_ep5.pt'))
dec_eval.load_state_dict(torch.load('decoder_ep5.pt'))

# 4. Set to Evaluation Mode
# This turns off Dropout and tells the LSTM to stop calculating gradients
enc_eval.eval()
dec_eval.eval()

print("Models loaded and ready for inference!")

/home/aakash/.conda/envs/gpu_env/lib/python3.10/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Models loaded and ready for inference!


# Training SANTA (SANskrit Translating Agent)

In [7]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import os
import random
import torch
import gc
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    TrainerCallback  
)

# 1. Hardware & ICARUS Setup

# ICARUS weights (enc_eval, dec_eval) and char maps are already in memory
def icarus_morphological_analyzer(input_string, max_len=256):
    # Transliterate Devanagari to SLP1 for Model 1
    ascii_text = transliterate(input_string, sanscript.DEVANAGARI, sanscript.SLP1)
    
    # ICARUS Inference logic
    input_ids = [char_to_ix.get(char, char_to_ix["<UNK>"]) for char in ascii_text]
    input_tensor = torch.LongTensor(input_ids).unsqueeze(1).to(device)
    
    results = []
    with torch.no_grad():
        encoder_outputs, (hidden, cell) = enc_eval(input_tensor)
        decoder_input = torch.tensor([char_to_ix["<SOS>"]]).to(device)
        for _ in range(max_len):
            prediction, hidden, cell = dec_eval(decoder_input, hidden, cell, encoder_outputs)
            top_idx = prediction.argmax(1).item()
            if top_idx == char_to_ix["<EOS>"]: break
            results.append(ix_to_char.get(top_idx, ""))
            decoder_input = torch.tensor([top_idx]).to(device)
            
    return "".join(results) # Returns: "rAma <p=n,c=1> | ..."


# 2. Data Preprocessing (Stage 1: Segmentation)

print("Loading Samanantar subset (20k rows)...")
dataset = load_from_disk("/home/aakash/akshat/local_bpcc_dataset")
#print(dataset['tgt_lang'])
small_dataset = dataset.shuffle(seed=42).select(range(20000))

def apply_icarus(example):
    # Get the segmented/tagged version using ICARUS
    # 'tgt' is the Sanskrit column in Samanantar
    example["segmented_sanskrit"] = icarus_morphological_analyzer(example["tgt"])
    return example

print("Running ICARUS Morphological Analysis...")
segmented_dataset = small_dataset.map(apply_icarus, desc="ICARUS Processing")

# --- FREE GPU MEMORY FOR INDICBART ---
del enc_eval
del dec_eval
gc.collect()
torch.cuda.empty_cache()
print("ICARUS cleared from GPU. Ready for IndicBART.")


# Model 2: IndicBART Training
# ---------------------------------------------------------
local_model_path = "/home/aakash/akshat/local_indicbart"
tokenizer = AutoTokenizer.from_pretrained(local_model_path, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path).to(device)

def preprocess_for_bart(examples):
    # Now IndicBART sees the hints from ICARUS
    inputs = [f"{ex} </s> <2san>" for ex in examples["segmented_sanskrit"]]
    targets = [f"<2en> {ex} </s>" for ex in examples["src"]]

    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing for IndicBART...")
tokenized_datasets = segmented_dataset.train_test_split(test_size=2000).map(
    preprocess_for_bart, batched=True, remove_columns=segmented_dataset.column_names
)

KeyboardInterrupt: 

In [47]:
segmented_dataset

Dataset({
    features: ['tgt', 'src', 'src_lang', 'tgt_lang', 'segmented_sanskrit'],
    num_rows: 20000
})

In [ ]:
# Save the 20k processed dataset
save_path = "/home/aakash/akshat/icarus_segmented_20k"

segmented_dataset.save_to_disk(save_path)
print(f"Dataset successfully saved to {save_path}")

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset successfully saved to /home/aakash/akshat/icarus_segmented_20k


In [55]:
segmented_dataset['src']

Column(['He also established an efficient postal system, with mail being carried by relays of horse riders.', 'The King of Hedamba having no heir made the eldest son of Trilochona, the King of Hedamba.', 'One day, Sringeri Srinivas, the banana farmer, came home from the cattle fair with a new cow. “We will call her Laxmi,” said his wife, Parvatamma.', 'Hindus represent the biggest religious group in all districts except Malappuram, where they are outnumbered by Muslims.', 'Several different designs of man-lifting kites were developed.', ...])

In [12]:
import re
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def convert_to_devanagari_safe(text):
    # Regex logic: 
    # Find segments that are NOT inside < >
    # Example: "rAma <p=n> | vana <p=n>" 
    # Result: rAma -> राम, vana -> वन, but <p=n> stays <p=n>
    
    def replace_match(match):
        word = match.group(0)
        # Only transliterate if it's not a tag or a separator like '|'
        if word.strip() == "|" or word.startswith("<"):
            return word
        return transliterate(word, sanscript.SLP1, sanscript.DEVANAGARI)

    # This pattern catches either a tag <...> OR sequences of non-bracket characters
    pattern = r"(<[^>]+>|[^<>|]+|\|)"
    
    segments = re.findall(pattern, text)
    converted_segments = []
    
    for seg in segments:
        if seg.startswith("<") or seg == "|":
            converted_segments.append(seg)
        else:
            # Transliterate the actual Sanskrit words
            converted_segments.append(transliterate(seg, sanscript.SLP1, sanscript.DEVANAGARI))
            
    return "".join(converted_segments)

In [57]:
import re
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def finalize_for_indicbart(example):
    # Convert the ICARUS SLP1 output to Devanagari while preserving tags
    raw_icarus = example["segmented_sanskrit"]
    example["segmented_devanagari"] = convert_to_devanagari_safe(raw_icarus)
    return example

# Apply the conversion to the entire 20k dataset
# Using num_proc makes this much faster
segmented_dataset = segmented_dataset.map(finalize_for_indicbart, num_proc=4)

Map (num_proc=4):   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
# Save the 20k processed dataset
save_path = "/home/aakash/akshat/icarus_segmented_20k"

segmented_dataset.save_to_disk(save_path)
print(f"Dataset successfully saved to {save_path}")

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset successfully saved to /home/aakash/akshat/icarus_segmented_20k


In [ ]:
import os
import random
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    TrainerCallback  
)

# ---------------------------------------------------------
# Custom Callback for Mid-Epoch Translation
# ---------------------------------------------------------
class MidEpochTranslationCallback(TrainerCallback):
    """
    Runs the exact inference logic on sample sentences at the end of every epoch.
    Also prints a training sample at the start of every epoch.
    """
    def __init__(self, tokenizer, test_sentences, device, train_dataset):
        self.tokenizer = tokenizer
        self.test_sentences = test_sentences
        self.device = device
        self.train_dataset = train_dataset

    def on_epoch_begin(self, args, state, control, **kwargs):
        # Pick a random sample from the training dataset
        sample = random.choice(self.train_dataset)
        
        # Replace -100 labels with pad_token_id 
        labels = [l if l != -100 else self.tokenizer.pad_token_id for l in sample["labels"]]
        
        # Decode the raw token IDs to show exactly what goes into the model
        input_str = self.tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
        target_str = self.tokenizer.decode(labels, skip_special_tokens=False)
        
        print("\n" + "*"*40)
        print(f"TRAINING DATA SAMPLE (Epoch {state.epoch:.1f} Start)")
        print(f"Model sees Input : {input_str}")
        print(f"Model sees Target: {target_str}")
        print("*"*40 + "\n")

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        print("\n" + "="*40)
        print(f"MID-EPOCH TEST (Epoch {state.epoch:.1f}). RUNNING TRANSLATIONS...")
        print("="*40)
        
        # Set model to evaluation mode
        model.eval()
        
        for i, sanskrit_text in enumerate(self.test_sentences):
    
            formatted_input = f"{sanskrit_text} </s> <2san>"
            
    
            inputs = self.tokenizer(
                formatted_input, 
                add_special_tokens=False, 
                return_tensors="pt", 
                max_length=128, 
                truncation=True
            ).to(self.device)

            with torch.no_grad():
                
                # Since the tag is split, we tokenize "<2en>" to get the exact sequence of token IDs
                forced_prefix_ids = self.tokenizer("<2en>", add_special_tokens=False, return_tensors="pt").input_ids.to(self.device)
                
                # Fetch the standard decoder start token (usually <s> or </s> depending on the model)
                start_id = model.config.decoder_start_token_id
                if start_id is None:
                    start_id = self.tokenizer.bos_token_id if self.tokenizer.bos_token_id is not None else self.tokenizer.eos_token_id
                
        
                decoder_input_ids = torch.cat(
                    [torch.tensor([[start_id]], device=self.device), forced_prefix_ids],
                    dim=1
                )
                
                outputs = model.generate(
                    **inputs,
                    max_length=128,
                    num_beams=5,             
                    repetition_penalty=3.5,  # Increased penalty to force the model away from Sanskrit
                    no_repeat_ngram_size=3,  # Prevents looping
                    early_stopping=True,
                    decoder_input_ids=decoder_input_ids 
                )
            
            # Decode the output
            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Cleanup any stray characters from the split tag <2en>
            translation = translation.replace("<2en>", "").replace("<", "").replace("2en>", "").replace("/s>", "").strip()
            
            print(f"Sanskrit : {sanskrit_text}")
            print(f"English  : {translation}")
            print("-" * 40)
            
        # Switch back to training mode
        model.train()

# ---------------------------------------------------------
# Hardware Setup
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Initializing Offline Training Pipeline on: {device.upper()}")

# POINT TO THE LOCAL FOLDERS 
local_model_path = "/home/aakash/akshat/local_indicbart"
#local_dataset_path = "/home/aakash/akshat/local_bpcc_dataset"

# 2. Load the Tokenizer 
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(local_model_path, use_fast=True, local_files_only=True)

# Bypass normalizer to preserve Sanskrit Matras
tokenizer.backend_tokenizer.normalizer = None

# 3. Load the Model (From Local Directory)
print("Loading Model...")
model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path, local_files_only=True).to(device)

# Silence "tied weights" warning to ensure clean gradients
#model.config.tie_word_embeddings = False

# We must let the model use its default start token to learn properly.

# ---------------------------------------------------------
# Load & Prepare the ICARUS-Processed Dataset which will train SANTA
# ---------------------------------------------------------
print("Loading ICARUS-Segmented Dataset from local disk...")

icarus_dataset_path = "/home/aakash/akshat/icarus_segmented_20k" 
dataset = load_from_disk(icarus_dataset_path)

# Split
print("Splitting into 18k train / 2k validation...")
split_datasets = dataset.train_test_split(test_size=2000, seed=42)

# preprocess 
def preprocess_function(examples):
    processed_inputs = []
    for raw_output in examples["segmented_sanskrit"]:
        # Convert SLP1 stems back to Devanagari, leave tags as ASCII
        devanagari_input = convert_to_devanagari_safe(raw_output)
        
        # Format for IndicBART: "Sanskrit </s> <2san>"
        processed_inputs.append(f"{devanagari_input} </s> <2san>")
    
    # Target: "<2en> English text </s>"
    targets = [f"<2en> {ex} </s>" for ex in examples["src"]]

    model_inputs = tokenizer(processed_inputs, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing the segmented dataset...")
tokenized_datasets = split_datasets.map(
    preprocess_function, 
    batched=True, 
    remove_columns=split_datasets["train"].column_names
)

# Update Callback Test Sentences
# Since the model is learning [Segmented Devanagari + Tags], 
# the test sentences must be in that format for the callback to work.
test_sentences = [
    "राम <p=n,c=1> | वन <p=n,c=2> | गम् <p=v,c=nan>", 
    "भारत <p=n,c=1> | अस्मद् <p=p,c=6> | देश <p=n,c=1>",
    "सर्व <p=a,c=1> | भू <p=v,c=nan> | सुखिन् <p=a,c=1>"
]

translation_callback = MidEpochTranslationCallback(
    tokenizer=tokenizer,
    test_sentences=test_sentences,
    device=device,
    train_dataset=tokenized_datasets["train"] 
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Aggressive Training Arguments 
print("Setting up Trainer...")
training_args = Seq2SeqTrainingArguments(
    output_dir="./sanskrit_translator_v2",
    eval_strategy="epoch",
    save_strategy="epoch",           # Save weights at the end of each epoch
    learning_rate=1e-4,              # learning rate to force learning
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   
    weight_decay=0.01,
    save_total_limit=2,              # Keeps only the 2 most recent epochs to save disk space
    num_train_epochs=10,             # Increased epochs for deep learning
    predict_with_generate=False,     # Keep False to speed up training
    fp16=False,                      
    logging_steps=50,
    report_to="none"
)

# ICARUS + Tag format
test_sentences = [
    "राम <p=n,c=1> | वन <p=n,c=2> | गम् <p=v,c=nan>", 
    "भारत <p=n,c=1> | अस्मद् <p=p,c=6> | देश <p=n,c=1>",
    "सर्व <p=a,c=1> | भू <p=v,c=nan> | सुखिन् <p=a,c=1>"
]

translation_callback = MidEpochTranslationCallback(
    tokenizer=tokenizer,
    test_sentences=test_sentences,
    device=device,
    train_dataset=tokenized_datasets["train"] 
)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[translation_callback]  
)

# START TRAINING
print(" Starting Offline Training Loop...")
model.tie_weights() # Ensures embeddings are linked and saved together
trainer.train()

# Save the final model
final_path = "./sanskrit_translator_final_v2"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f" Model successfully trained and saved to {final_path}!")

 Initializing Offline Training Pipeline on: CUDA
Loading Tokenizer...
Loading Model...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Loading ICARUS-Segmented Dataset from local disk...
Splitting into 18k train / 2k validation...
Tokenizing the segmented dataset...
Setting up Trainer...


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


 Starting Offline Training Loop...

****************************************
TRAINING DATA SAMPLE (Epoch 0.0 Start)
Model sees Input : [CLS] दुष् <p=va,c=N<UNK>आ + तम <p=n,c=1> | तथा <p=i,c=nan> | सर्वोत्तम <p=n,c=1> | द्वि <p=n,c=6> | अपि <p=i,c=nan> | परिदृश् <p=va,c=6> | सन्दर्भ<p=va,c=1> | चिन्तय् <p=vi,c=nan> | प्रयत् <p=v,c=nan></s> <2san>[SEP]<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Model s

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


****************************************
TRAINING DATA SAMPLE (Epoch 9.0 Start)
Model sees Input : [CLS] अनन्तर <p=a,c=1> | जोशिन् <p=a,c=1> | इत्यय <p=n,c=2> | अभिनय <p=n,c=5> | विरम् <p=vi,c=nan> | सञ्जीव् <p=va,c=1> | भट्टाचार्य <p=n,c=6> | काम्प <p=n,c=1> | इत्यन <p=n,c=3> | कार्य <p=n,c=N<UNK>आ + क्रम <p=n,c=3> | युवनट <p=n,c=N<UNK>आ + रूप <p=n,c=3> |</s> <2san>[SEP]<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Model sees Target: [CLS]<2en> Later, Joshi took a break from acting and made a comeback as a youth actor with Sanjeev Bhattacharya's show Campus.</s>[SEP]<pad><pad><pad><pad><pad><pad>

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model successfully trained and saved to ./sanskrit_translator_final_v2!


In [15]:
import os

# Create a final directory
save_path = "/home/aakash/akshat/FINAL_MODEL_SANTA"
os.makedirs(save_path, exist_ok=True)

# 1. Force tie weights before saving
model.tie_weights()

# 2. Save the model using the base class (this is more reliable than Trainer)
model.save_pretrained(save_path, safe_serialization=False)

# 3. Save the tokenizer (crucial for testing)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}. Check the folder for 'pytorch_model.bin' (~900MB-1GB)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /home/aakash/akshat/FINAL_MODEL_SANTA. Check the folder for 'pytorch_model.bin' (~900MB-1GB)


In [27]:
# Define your path
dataset_path = "/home/aakash/akshat/sanskrit_split_dataset"

# Save the entire DatasetDict (train and test)
split_datasets.save_to_disk(dataset_path)

print(f"Dataset successfully saved to {dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/18000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset successfully saved to /home/aakash/akshat/sanskrit_split_dataset


# Metrics

In [33]:
import os

# FORCE OFFLINE MODE - Must be at the very top
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

import torch
from transformers import AutoTokenizer, AutoModel

# Path to the model you saved in Step 1
local_sbert_path = '/home/aakash/akshat/sbert_model'

# This should now work without the 'Name or service not known' error
#tokenizer = AutoTokenizer.from_pretrained(local_sbert_path, local_files_only=True)
sim_model = AutoModel.from_pretrained(local_sbert_path, local_files_only=True).to('cuda')
sim_tokenizer = AutoTokenizer.from_pretrained(local_sbert_path, local_files_only=True)

print("✅ Success! SBERT loaded in Offline Mode.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Success! SBERT loaded in Offline Mode.


In [34]:
# function to infer using ICARUS
def predict(input_string, max_len=500):
    # Prepare input: Map chars to IDs and move to GPU
    input_ids = [char_to_ix.get(char, char_to_ix["<UNK>"]) for char in input_string]
    input_tensor = torch.LongTensor(input_ids).unsqueeze(1).cuda() # Shape: [seq_len, 1]
    
    results = []
    
    with torch.no_grad():
        # Encode the full input
        encoder_outputs, (hidden, cell) = enc_eval(input_tensor)
        
        # Prepare Decoder's first input (<SOS>)
        decoder_input = torch.tensor([char_to_ix["<SOS>"]]).cuda()
        
        # Loop character by character
        for _ in range(max_len):
            prediction, hidden, cell = dec_eval(decoder_input, hidden, cell, encoder_outputs)
            
            # Get the index with highest probability
            top_idx = prediction.argmax(1).item()
            
            # If the model predicts End of Sentence, stop
            if top_idx == char_to_ix["<EOS>"]:
                break
            
            # Map back to char and store
            results.append(ix_to_char.get(top_idx, ""))
            
            # Use this prediction as the input for the next time step
            decoder_input = torch.tensor([top_idx]).cuda()
            
    return "".join(results)

In [ ]:
def get_segmentation(example):
    text = example["ascii_sanskrit"]
    
    if not text or text.strip() == "":
        return {"segmented_sanskrit": ""}
    
    # Run the prediction
    # We increase max_len if the input is long to avoid cutting off
    full_prediction = predict(text, max_len=len(text) * 10) 
    
    # We keep the WHOLE thing because the pipes are your word separators
    return {"segmented_sanskrit": full_prediction}

#dataset = dataset.map(get_segmentation)

Map:   0%|          | 0/391533 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
import re
import torch
import sacrebleu
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from datasets import load_from_disk


def generate_translation(batch_texts):
    # 1. Format and Tokenize
    formatted_inputs = [f"{text} </s> <2san>" for text in batch_texts]
    inputs = tokenizer(formatted_inputs, return_tensors="pt", padding=True, truncation=True, max_length=256)
    
    # 2. Extract and force to GPU
    # Explicitly using .to(device, non_blocking=True) helps on some clusters
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # 3. Create Decoder sequence on GPU
    # We must ensure the 'forced_prefix_ids' and 'eos_tensor' are on the same device as input_ids
    forced_prefix_ids = tokenizer("<2en>", add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    
    eos_id = tokenizer.convert_tokens_to_ids('</s>')
    # Create the tensor directly on the GPU
    eos_tensor = torch.full((len(batch_texts), 1), eos_id, dtype=torch.long, device=device)
    
    # Concatenate - this result will be on the GPU
    decoder_input_ids = torch.cat([eos_tensor, forced_prefix_ids.repeat(len(batch_texts), 1)], dim=1)

    # 4. Final Generate Call
    # We explicitly pass ONLY the tensors we just moved to GPU
    with torch.no_grad():
        generated_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            max_length=128,
            num_beams=5,
            early_stopping=True,
            use_cache=True # Helps with speed
        )
    
    # 5. Decode
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return [p.replace("<2en>", "").strip() for p in decoded_preds]


# 1. Setup - Path to saved
#model_path = "/home/aakash/akshat/FINAL_MODEL_SANTA"
#model = MBartForConditionalGeneration(config)
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset_path = "/home/aakash/akshat/sanskrit_split_dataset"
split_datasets = load_from_disk(dataset_path)

# 2. Cleaning Function: Removes any Sanskrit/Devanagari left in the English output
def clean_output(text):
    # Removes Devanagari block [\u0900-\u097F]
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    # Removes extra special tokens/whitespace
    text = text.replace('</s>', '').replace('<2en>', '').strip()
    return " ".join(text.split())

# 3. Load Semantic Model (Small & Fast for Clusters)
# 'all-MiniLM-L6-v2' is only ~80MB but excellent for similarity
#sim_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)

def run_full_evaluation(split_datasets, generate_fn):
    test_data = split_datasets["test"]
    raw_preds = []
    references = [t for t in test_data["tgt"]]
    
    print("Running Inference on 2,000 test rows...")
    batch_size = 16
    for i in tqdm(range(0, len(test_data), batch_size)):
        batch_inputs = test_data["segmented_devanagari"][i : i + batch_size]
        batch_refs = test_data["src"][i : i + batch_size]
        
        preds = generate_fn(batch_inputs) 
        raw_preds.extend(preds)

        # Print the last 3 from the current batch
        # We'll do this every batch, but you can add 'if i % 160 == 0:' to see fewer
        if i%10 ==0:
            print(f"\n--- Samples from Batch {i//batch_size + 1} ---")
            for j in range(-3, 0): # Last 3 items
                try:
                    print(f"REF: {batch_refs[j]}")
                    print(f"PRED: {preds[j]}")
                    print(f"INPUT: {batch_inputs[j]}")
                    print("-" * 15)
                except IndexError:
                    pass # Handles the final batch if it's smaller than 3

    # Clean the predictions for evaluation
    cleaned_preds = [clean_output(p) for p in raw_preds]

    # --- Metric 1: BLEU Score ---
    bleu = sacrebleu.corpus_bleu(cleaned_preds, [references])
    
    # --- Metric 2: Semantic Similarity (Manual Mean Pooling) ---
    print("🧠 Calculating Semantic Similarity...")

    def get_embeddings(texts, model, tokenizer):
        # Tokenize
        encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(device)
        # Compute token embeddings
        with torch.no_grad():
            model_output = model(**encoded_input)
        # Perform pooling. In this case, mean pooling.
        token_embeddings = model_output[0] # First element of model_output contains all token embeddings
        attention_mask = encoded_input['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # Get embeddings for both
    # Note: If memory is an issue, you can do this in smaller batches
    emb_preds = get_embeddings(cleaned_preds, sim_model, sim_tokenizer)
    emb_refs = get_embeddings(references, sim_model, sim_tokenizer)
    
    # Normalize for cosine similarity
    emb_preds = torch.nn.functional.normalize(emb_preds, p=2, dim=1)
    emb_refs = torch.nn.functional.normalize(emb_refs, p=2, dim=1)
    
    # Pairwise cosine similarity (dot product of normalized vectors)
    cosine_scores = (emb_preds * emb_refs).sum(dim=1)
    avg_sim = cosine_scores.mean().item()
    pattern = r'[\u0900-\u097F]'
    # 4. Final Report
    print(f"\n{'='*40}")
    print(f"📊 EVALUATION REPORT")
    print(f"{'='*40}")
    print(f"BLEU Score:         {bleu.score:.2f}")
    print(f"Semantic Similarity: {avg_sim:.4f} (0 to 1 scale)")
    print(f"Sanskrit Leaks:      {sum(1 for p in raw_preds if re.search(pattern, p))/len(raw_preds)*100:.1f}%")
    print(f"{'='*40}")
    
    return cleaned_preds

# Execute
print(f"Model is on: {next(model.parameters()).device}")
print(f"Device variable is: {device}")
final_predictions = run_full_evaluation(split_datasets, generate_translation)

Model is on: cuda:0
Device variable is: cuda
Running Inference on 2,000 test rows...


  1%|▋                                                                                  | 1/125 [00:02<05:12,  2.52s/it]


--- Samples from Batch 1 ---
REF: Utkarsa was in his turn imprisoned and he committed suicide.
PRED: [unused1] < 2en > casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casually casuall

  5%|███▉                                                                               | 6/125 [00:14<04:55,  2.49s/it]


--- Samples from Batch 6 ---
REF: I would like to see the Punjab, North-West Frontier Province, Sind and Baluchistan, amalgamated into a single State.
PRED: [unused1] < 2en > she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she
INPUT: अहर् <p=n,c=1> | पञ्जाब्<p=n,c=N<UNK>आ + बुराज्य <p=n,c=2> | वायव्य <p=a,c=N<UNK>आ + सीमा <p=n,c=N<UNK>आ + प्रम्त <p=n,c=2> | सिन्धि <p=n,c=2> | बलू <p=n,c=N<UNK>आ + चिस्त <p=n,c=2> | च <p=i,c=nan> | एकराज्य <p=n,c=N<UNK>आ + त्व <p=n,c=3> | समाहित <p=a,c=2> | दृश् प्=
---------------
REF: Two of the most important tree varietie

...


--- Samples from Batch 16 ---
REF: The most common symptoms of Alzheimer's disease are short-term memory loss and word-finding difficulties.
PRED: [unused1] < 2en > she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she she
INPUT: अल्प <p=a,c=N<UNK>आ + कालिक <p=n,c=N<UNK>आ + स्मृति <p=n,c=N<UNK>आ + क्षय <p=n,c=1> | शब्द <p=n,c=N<UNK>आ + निर्णय <p=n,c=7> | कठिनता <p=n,c=1> | च <p=i,c=nan> | इत् <p=v,c=nan> | अल्ज् <p=va,c=N<UNK>आ + मरर्<p=n,c=N<UNK>आ + ओग <p=n,c=6> | त्यन्त<p=v,c=nan> | सामान्य 
---------------
REF: For instance, you can choose to focus on relaxing

 17%|█████████████▊                                                                    | 21/125 [00:52<04:17,  2.48s/it]


--- Samples from Batch 21 ---
REF: The state hosts many religious sects such as Satnampanth, Kabirpanth, Ramnami Samaj, and others.
PRED: [unused1] < 2en > long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long
INPUT: इदम् <p=n,c=7> | राज्य <p=n,c=7> | सत् <p=a,c=N<UNK>आ + अनापन्थ <p=n,c=1> | कबीर <p=n,c=N<UNK>आ + पथिन् <p=n,c=1> | राम <p=n,c=N<UNK>आ + नामी <p=n,c=N<UNK>आ + समाज <p=n,c=1> | इत्यादय <p=n,c=1> | अनेक <p=a,c=7> | धार्मिक <p=n,c=N<UNK>आ + सम्प्रदा <p=n,c=6> | प्रच्
---------------
REF: Gandhi was elected to the Lok Sabha from the Raeba

 18%|██████████████▍                                                                   | 22/125 [00:55<04:18,  2.51s/it]


KeyboardInterrupt: 